In [1]:
import pandas as pd
import ast

In [14]:
df = pd.read_csv(r"D:\Thesis EGFR Round 3\Dataset\concatenated_cdrs.csv")
print(len(df))

206031


In [18]:
unique_cdrs = df["Concatenated_CDRs"].unique()
print("Number of unique CDRs:", len(unique_cdrs))
seq_counts = df["Concatenated_CDRs"].value_counts()
print(seq_counts)

Number of unique CDRs: 182693
Concatenated_CDRs
GSIFSGSFISDGASTAAVLYPPYPYDY            553
GRSFSTYAISWTDSTAADRWASSRRNVDYDY        202
GTISYSDYIDGGASTAVLQVYPYHQY             201
GYISQERTIDKGSNTAALSRSRAGYAY            180
GNISEIVDITRGSITAVDEYDGTRSLGY           171
                                      ... 
GTIFADYDIAYGSSTAVWSRRTIRWHDWSIRNYAY      1
GYISGQGWITAGGTTAARSHALTAYVY              1
GNISVGSTIAYGASTVVSYSVAYWHGVDLHPCHYY      1
GYISGPNGIGFGGSTAAATRLHDSYDY              1
GTISWWSISLGATTAVDPYSTPVSHGEHQY           1
Name: count, Length: 182693, dtype: int64


In [4]:
seq_label_counts = (
    df.groupby("Concatenated_CDRs")["Label"]
      .nunique()
      .reset_index(name="n_labels")
)

In [7]:
sequences_in_multiple_labels = seq_label_counts[
    seq_label_counts["n_labels"] > 1
]
print("Sequences appearing in more than one label:")
print(sequences_in_multiple_labels)

Sequences appearing in more than one label:
                      Concatenated_CDRs  n_labels
882         DSISGYNTIAAGGITAVWDTYDSGYGY         2
1782        GNIFAAAYISRGATTAALISYNRVLDY         2
1880    GNIFAAYGINHGTTTAVTSVRRVANPLDLAY         2
1948        GNIFADNDIDFGSTTAADLRTGADYPY         2
2056        GNIFAFLSIARGASTAVAYGSSGPYQY         2
...                                 ...       ...
181593  GYISYWTQIGTGANTAARVRLWYADRYDHQY         2
181880      GYISYYSSIDGGSSTAAFDYYESPFGY         2
181953      GYISYYWDIDWGSNTAADWWDGGGYYY         2
182076      GYISYYYAIGAGGSTAVWRQYPDRHYY         2
182445        SIFSRGYIGSGTNTAAYGVSPSLGY         2

[2250 rows x 2 columns]


In [8]:
problem_seq_to_label = (
    df[df["Concatenated_CDRs"].isin(sequences_in_multiple_labels["Concatenated_CDRs"])]
    .groupby("Concatenated_CDRs")["Label"]
    .apply(lambda x: sorted(set(x)))
    .reset_index(name="labels")
)

In [ ]:
print("Sequences with multiple labels:")
print(problem_seq_to_label)
problem_seq_to_label.to_csv("D:\Thesis EGFR Round 3\Dataset\problem_seq_to_label.csv", index=False)

Sequences with multiple labels:
                    Concatenated_CDRs  labels
0         DSISGYNTIAAGGITAVWDTYDSGYGY  [0, 1]
1         GNIFAAAYISRGATTAALISYNRVLDY  [0, 1]
2     GNIFAAYGINHGTTTAVTSVRRVANPLDLAY  [0, 1]
3         GNIFADNDIDFGSTTAADLRTGADYPY  [0, 1]
4         GNIFAFLSIARGASTAVAYGSSGPYQY  [0, 1]
...                               ...     ...
2245  GYISYWTQIGTGANTAARVRLWYADRYDHQY  [0, 1]
2246      GYISYYSSIDGGSSTAAFDYYESPFGY  [0, 1]
2247      GYISYYWDIDWGSNTAADWWDGGGYYY  [0, 1]
2248      GYISYYYAIGAGGSTAVWRQYPDRHYY  [0, 1]
2249        SIFSRGYIGSGTNTAAYGVSPSLGY  [0, 1]

[2250 rows x 2 columns]


In [ ]:
dup_split = (
    df.groupby("Concatenated_CDRs")
    .agg(
        n_records=("Nanobody_id", "count"),
        n_labels=("Label", "nunique")
    )
    .reset_index()
)

dup_split = dup_split[(dup_split["n_records"] > 1) & (dup_split["n_labels"] > 1)]
print("Duplicated sequences split across labels")
print(dup_split)
print(dup_split["n_records"].sum())


=== Duplicated sequences split across labels ===
                      Concatenated_CDRs  n_records  n_labels
882         DSISGYNTIAAGGITAVWDTYDSGYGY          2         2
1782        GNIFAAAYISRGATTAALISYNRVLDY          3         2
1880    GNIFAAYGINHGTTTAVTSVRRVANPLDLAY          2         2
1948        GNIFADNDIDFGSTTAADLRTGADYPY          2         2
2056        GNIFAFLSIARGASTAVAYGSSGPYQY          2         2
...                                 ...        ...       ...
181593  GYISYWTQIGTGANTAARVRLWYADRYDHQY          2         2
181880      GYISYYSSIDGGSSTAAFDYYESPFGY          2         2
181953      GYISYYWDIDWGSNTAADWWDGGGYYY        131         2
182076      GYISYYYAIGAGGSTAVWRQYPDRHYY          2         2
182445        SIFSRGYIGSGTNTAAYGVSPSLGY          2         2

[2250 rows x 3 columns]
5270


In [ ]:
dup_split = (
    df.groupby("Concatenated_CDRs")
    .agg(
        n_records=("Nanobody_id", "count"),
        n_labels=("Label", "nunique"),
        n_label_0=("Label", lambda x: (x == 0).sum()),
        n_label_1=("Label", lambda x: (x == 1).sum())
    )
    .reset_index()
)

dup_split = dup_split[(dup_split["n_records"] > 1) & (dup_split["n_labels"] > 1)]

print(" Duplicated sequences split across labels")
print(dup_split)
dup_split.to_csv(r"D:\Thesis EGFR Round 3\Dataset\duplicated_sequences_labels.csv", index=False)


=== Duplicated sequences split across labels ===
                      Concatenated_CDRs  n_records  n_labels  n_label_0  \
882         DSISGYNTIAAGGITAVWDTYDSGYGY          2         2          1   
1782        GNIFAAAYISRGATTAALISYNRVLDY          3         2          1   
1880    GNIFAAYGINHGTTTAVTSVRRVANPLDLAY          2         2          1   
1948        GNIFADNDIDFGSTTAADLRTGADYPY          2         2          1   
2056        GNIFAFLSIARGASTAVAYGSSGPYQY          2         2          1   
...                                 ...        ...       ...        ...   
181593  GYISYWTQIGTGANTAARVRLWYADRYDHQY          2         2          1   
181880      GYISYYSSIDGGSSTAAFDYYESPFGY          2         2          1   
181953      GYISYYWDIDWGSNTAADWWDGGGYYY        131         2          1   
182076      GYISYYYAIGAGGSTAVWRQYPDRHYY          2         2          1   
182445        SIFSRGYIGSGTNTAAYGVSPSLGY          2         2          1   

        n_label_1  
882             1  
1782     

In [34]:
cdrs_df = pd.read_csv(r"D:\Thesis EGFR Round 3\Dataset\concatenated_cdrs.csv")
print(len(cdrs_df))

206031


In [35]:
dup_labels_df = pd.read_csv(r"D:\Thesis EGFR Round 3\Dataset\duplicated_sequences_labels.csv")
print(len(dup_labels_df))

2250


In [36]:
cdrs_df.loc[cdrs_df["Concatenated_CDRs"].isin(dup_labels_df["Concatenated_CDRs"]), "Label"] = 1
print(len(cdrs_df))

206031


In [39]:
print(cdrs_df)
cdrs_df.to_csv(r"D:\Thesis EGFR Round 3\Dataset\crosstable_seq_to_label.csv", index=False)

            Nanobody_id                    Concatenated_CDRs  Label
0       Nb_e3fe0d0da712          GNIFPADTIDDGTTTAVNYRYDNYLYY      0
1       Nb_da3f82c93dc0  GYIFATTDIDFGSSTAVGTRPYTRSSDYAEHGFEY      0
2       Nb_8f59de1c0110      GYISNQYGIGYGGNTAAVIGPDSRVLRDFDY      0
3       Nb_594e2f2137ac          GNISPIQFINYGTNTAAEGYGIPYHSY      0
4       Nb_ed3829266cd4            GTISWVGHINAGGTTAAANQYYHPY      0
...                 ...                                  ...    ...
206026  Nb_ab9bd88afe7c          GYIFIQSRITEGGTTAADARRDTSLPY      1
206027  Nb_717fda807fa1       GTISPRKSIAIGGSTAAYRGYPYDQRYHGY      1
206028  Nb_1627e4b49b87          GTISYSDYIDGGASTAALSRSRAGYAY      1
206029  Nb_a84856fa6340           GNISEYAIDRGGSTAAVLYPPYPYDY      1
206030  Nb_4099c0154887       GTIFAGYIDVGSTTAASYYNERYTQLDYVY      1

[206031 rows x 3 columns]


In [ ]:
cdrs_df = pd.read_csv(r"D:\Thesis EGFR Round 3\Dataset\Clustering\crosstable_seq_to_label.csv")
seq_summary = (
    cdrs_df.groupby(["Concatenated_CDRs", "Label"])
      .agg(
          Count=("Nanobody_id", "size"),
          Nanobody_ids=("Nanobody_id", lambda x: list(x))
      )
      .reset_index()
)
print(seq_summary)
seq_summary.to_csv(r"D:\Thesis EGFR Round 3\Dataset\Clustering\seq_summary1.csv", index=False)

                          Concatenated_CDRs  Label  Count       Nanobody_ids
0       ADISRGRYIGYGTSTAASWQVRATGQYERDTYFGY      0      1  [Nb_99992a92ba17]
1             AFQYAGIDRGTTTAAPFVPTGTYAYELYY      0      1  [Nb_9b31a8e81717]
2               AHIFRRAGINEVAITSADDYAPYVHLY      0      1  [Nb_53eaa84f339b]
3               AHISRVSVITDGGTTADGRSYDFWLYY      0      1  [Nb_6efb310f9a39]
4               AHISYYKYIGFVGSTSASYAPYSPHYY      0      1  [Nb_dc11fad11220]
...                                     ...    ...    ...                ...
182688       YISGYIYINDGANTAAAYPSGYGYYDYLEY      1      1  [Nb_20ce1375c339]
182689           YISPSYPIDGGGTTAAQRYTYLPYFY      0      1  [Nb_2a25f7cac796]
182690       YISPSYPIDRGGTTAVDEYDQNDGGYSYGY      1      1  [Nb_02bf5198f9dd]
182691           YISPSYPIDRGGTTAVLIARWQAHDY      0      1  [Nb_ce09f20373b6]
182692           YISVQNSISGGGNTAVRYLTRAEYKY      1      1  [Nb_758ef51784b8]

[182693 rows x 4 columns]


In [ ]:
seq_summary = pd.read_csv(r"D:\Thesis EGFR Round 3\Dataset\Clustering\seq_summary1.csv")
seq_summary["ID"]= range(1, len(seq_summary) + 1)
clustering = seq_summary[["ID","Concatenated_CDRs", "Label"]]
print(clustering)
clustering.to_csv(r"D:\Thesis EGFR Round 3\Dataset\Clustering\clustering_cdrs1.csv", index=False)

            ID                    Concatenated_CDRs  Label
0            1  ADISRGRYIGYGTSTAASWQVRATGQYERDTYFGY      0
1            2        AFQYAGIDRGTTTAAPFVPTGTYAYELYY      0
2            3          AHIFRRAGINEVAITSADDYAPYVHLY      0
3            4          AHISRVSVITDGGTTADGRSYDFWLYY      0
4            5          AHISYYKYIGFVGSTSASYAPYSPHYY      0
...        ...                                  ...    ...
182688  182689       YISGYIYINDGANTAAAYPSGYGYYDYLEY      1
182689  182690           YISPSYPIDGGGTTAAQRYTYLPYFY      0
182690  182691       YISPSYPIDRGGTTAVDEYDQNDGGYSYGY      1
182691  182692           YISPSYPIDRGGTTAVLIARWQAHDY      0
182692  182693           YISVQNSISGGGNTAVRYLTRAEYKY      1

[182693 rows x 3 columns]


Cluster list creation back with cross table

1. Take only first sequence if same CDR region (CDR1+CDR2+CDR3)

In [4]:
seq_sum = pd.read_csv(r"D:\Thesis EGFR Round 3\Dataset\Clustering\seq_summary1.csv")
print(len(seq_sum))
seq_sum["Nanobody_ids"] = seq_sum["Nanobody_ids"].apply(ast.literal_eval)
seq_sum["Nanobody_ids"] = seq_sum["Nanobody_ids"].apply(lambda x: x[0])
print(len(seq_sum))
seq_sum = seq_sum.drop(columns=["Count"])
seq_sum = seq_sum.rename(columns={"Nanobody_ids": "Nanobody_id"})
seq_sum.to_csv(r"D:\Thesis EGFR Round 3\Dataset\Clustering\changed_seq_summary1.csv", index=False)

182693
182693


In [ ]:
clusters = pd.read_csv(r"D:\Thesis EGFR Round 3\Dataset\Clustering\70\unique_cdhit_cluster_list701.csv")  # first file
nanobodies = pd.read_csv(r"D:\Thesis EGFR Round 3\Dataset\Clustering\changed_seq_summary1.csv")  # second file
#nanobodies["Nanobody_id"] = nanobodies["Nanobody_id"].apply(ast.literal_eval)

nanobodies = nanobodies.explode("Nanobody_id")
result = nanobodies.merge(
    clusters[["Cluster_number", "CDR_sequence"]],
    left_on="Concatenated_CDRs",
    right_on="CDR_sequence",
    how="left"
)
result = result[["Cluster_number", "Nanobody_id", "CDR_sequence", "Label"]]

#result = result.rename(columns={"Nanobody_id": "Nanobody_id"})
result.to_csv(r"D:\Thesis EGFR Round 3\Dataset\Clustering\70\changed_cluster_list_nanobody_mapping1.csv", index=False)
print(result.head())

   Cluster_number      Nanobody_id                         CDR_sequence  Label
0             702  Nb_99992a92ba17  ADISRGRYIGYGTSTAASWQVRATGQYERDTYFGY      0
1           55333  Nb_9b31a8e81717        AFQYAGIDRGTTTAAPFVPTGTYAYELYY      0
2           58296  Nb_53eaa84f339b          AHIFRRAGINEVAITSADDYAPYVHLY      0
3           58297  Nb_6efb310f9a39          AHISRVSVITDGGTTADGRSYDFWLYY      0
4           58298  Nb_dc11fad11220          AHISYYKYIGFVGSTSASYAPYSPHYY      0


In [3]:
print(len(result))

182693


In [4]:
result["CDR_sequence"].nunique()

182693

In [ ]:
print("Rows in clusters:", len(clusters))
print("Unique CDR_sequence in clusters:", clusters["CDR_sequence"].nunique())
dups = clusters[clusters.duplicated("CDR_sequence", keep=False)].sort_values("CDR_sequence")
print(dups.head(20))
print("Number of duplicated CDR sequences in clusters:", dups["CDR_sequence"].nunique())

Rows in clusters: 182693
Unique CDR_sequence in clusters: 182693
Empty DataFrame
Columns: [Cluster_number, ID, CDR_sequence, Is_representative]
Index: []
Number of duplicated CDR sequences in clusters: 0


In [6]:
cluster_list = pd.read_csv(r"D:\Thesis EGFR Round 3\Dataset\Clustering\70\unique_cdhit_cluster_list701.csv")
print((cluster_list["CDR_sequence"].nunique()))
print(len(cluster_list))

182693
182693


In [ ]:
seq_summary = pd.read_csv(r"D:\Thesis EGFR Round 3\Dataset\Clustering\changed_seq_summary1.csv")
print((seq_summary["Concatenated_CDRs"].nunique()))
print(len(seq_summary))

182693
182693


In [8]:
seq_label_counts = (
    seq_summary.groupby("Concatenated_CDRs")["Label"]
      .nunique()
      .reset_index(name="n_labels")
)
sequences_in_multiple_labels = seq_label_counts[
    seq_label_counts["n_labels"] > 1
]
print("Sequences appearing in more than one label:")
print(sequences_in_multiple_labels)

Sequences appearing in more than one label:
Empty DataFrame
Columns: [Concatenated_CDRs, n_labels]
Index: []


Check clustering

In [9]:
before_cluster = pd.read_csv(r"D:\Thesis EGFR Round 3\Dataset\Clustering\crosstable_seq_to_label.csv")
after_cluster = pd.read_csv(r"D:\Thesis EGFR Round 3\Dataset\Clustering\70\changed_cluster_list_nanobody_mapping1.csv")

print("Length of before clustering:", len(before_cluster))
print("Length of after clustering:", len(after_cluster))

print("Number of unique CDRs before clustering:", before_cluster["Concatenated_CDRs"].nunique())
print("Number of unique CDRs after clustering:", after_cluster["CDR_sequence"].nunique())

print("Distribution of labels before clustering:",before_cluster["Label"].value_counts())
print("Distribution of labels after clustering:",after_cluster["Label"].value_counts())

Length of before clustering: 206031
Length of after clustering: 182693
Number of unique CDRs before clustering: 182693
Number of unique CDRs after clustering: 182693
Distribution of labels before clustering: Label
1    122701
0     83330
Name: count, dtype: int64
Distribution of labels after clustering: Label
1    101379
0     81314
Name: count, dtype: int64
